# Train Time-As-Channels 3D CNN

This notebook intentionally keeps orchestration thin. Split preparation, fitting, reporting, plotting, and persistence are delegated to shared utilities in `src.ml`.


In [1]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from src.ml import (
    LossWeightConfig,
    OptimizationConfig,
    display_experiment_summary,
    display_holdout_evaluation,
    fit_estimator_on_experiment,
    persist_experiment_artifacts,
    plot_holdout_branch_embedding_projections,
    plot_training_history,
    prepare_multitask_experiment_data,
)
from src.dataset_config import load_current_dataset_artifact_path
from src.tensor_utils import build_tensor_embedding_2d, load_labeled_tensor_dataset, plot_tensor_embedding_2d

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [2]:
from src.ml import TimeChannel3DCNNClassifier, TimeChannel3DCNNConfig


In [3]:
# User inputs

dataset_artifact_path = load_current_dataset_artifact_path()
experiment_output_dir = Path("artifacts/nb6_time_channel_3dcnn")
persist_artifacts = True

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0

model_config = TimeChannel3DCNNConfig(
    conv_channels=(8, 16, 24, 32),
    kernel_size_z=(1, 1, 1, 1),
    kernel_size_xy=(5, 3, 3, 3),
    stride_z=(1, 1, 1, 1),
    stride_xy=(1, 1, 1, 1),
    pool_kernel_z=(1, 1, 1, 1),
    pool_kernel_xy=(2, 2, 2, 2),
    pool_stride_z=(1, 1, 1, 1),
    pool_stride_xy=(2, 2, 2, 2),
    embedding_dim=24,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=20,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=4,
    early_stopping_min_delta=0.0,
    scheduler_patience=1,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    compound_weight=0.2,
    concentration_weight=0.2,
)


In [4]:
dataset = load_labeled_tensor_dataset(dataset_artifact_path)


In [5]:
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)


In [6]:
display_experiment_summary(experiment)


,split,n_samples
0,train_augmented,245
1,train_base,35
2,val,9
3,holdout,15


,mechanism_of_action,compound,concentration_band,n_samples
0,NMDAR_Activation,(RS)-(Tetrazol-5-yl)glycine,control,42
1,GABAAR_NegativeAllostericModulator,DMCM,control,35
2,NMDAR_Antagonist,Ketamine,high,35
3,GABAAR_Antagonist,Bemegride,high,28
4,GABAAR_NegativeAllostericModulator,DMCM,high,28
5,NMDAR_Activation,(RS)-(Tetrazol-5-yl)glycine,high,28
6,NMDAR_Antagonist,Ketamine,control,28
7,GABAAR_Antagonist,Bemegride,control,21


In [7]:
model = TimeChannel3DCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)


In [ ]:
fit_estimator_on_experiment(model, experiment)


cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trA=train_action_loss
    trCC=train_commutative_consistency_loss
    trFA=train_feature_alignment_loss
    trCo=train_compound_loss
    trCn=train_concentration_loss
    vaL=val_loss
    vaA=val_action_loss
    vaCC=val_commutative_consistency_loss
    vaFA=val_feature_alignment_loss
    vaCo=val_compound_loss
    vaCn=val_concentration_loss
     ep       lr       eta |      trL      trA     trCC     trFA     trCo     trCn |      vaL      vaA     vaCC     vaFA     vaCo     vaCn
001/020 2.00e-04     04:02 |   2.1549   1.6671        -        -   1.6497   0.7891 |   1.9609   1.4985        -        -   1.6297   0.6821
002/020 2.00e-04     03:15 |   1.9735   1.5079        -        -   1.6015   0.7265 |   1.8940   1.4474        -        -   1.6051   0.6275
003/020 2.00e-04     02:53 |   1.9174   1.4575        -        -   1.5965   0.7030 |   1.8304   1.3932        -        -   1.5776   0.6087
004

In [ ]:
plot_training_history(model, title="Time-As-Channels 3D CNN loss curves", loess_frac=0.6);


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)


In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method="pca",
    random_state=optimization_config.random_state,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
)


In [ ]:
run_config = {
    "dataset_artifact_path": dataset_artifact_path,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}


In [ ]:
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
    )
